In [54]:
import pandas as  pd 
import json 

In [55]:
with open('candidates.jsonl', 'r') as f:
    first_line = f.readline()
    candidate = json.loads(first_line)

In [56]:
print(candidate)

{'candidate_id': 'CAND_0000001', 'profile': {'anonymized_name': 'Ira Vora', 'headline': 'Backend Engineer | SQL, Spark, Cloud', 'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", 'location': 'Toronto', 'country': 'Canada', 'years_of_experience': 6.9, 'current_title': 'Backend Engineer', 'current_company': 'Mindtree', 'current_company_size': '10001+', 'current_industry': 'IT Services'}, 'career_histo

In [58]:
def search_career_history_for_keywords(career_history, keywords):
    """
    Search through career history descriptions for keywords.
    Return True if ANY job contains ANY of the keywords.
    """
    for job in career_history:
        description = (job.get('description') or '').lower()  # Make it lowercase for searching
        for keyword in keywords:
            if keyword.lower() in description:
                return True
    return False
def search_skills_for_keywords(skills, keywords):
    """
    Search through skills list for keywords.
    Return True if ANY skill name contains ANY of the keywords.
    """
    for skill in skills:
        skill_name = (skill.get('name') or '').lower()
        for keyword in keywords:
            if keyword.lower() in skill_name:
                return True
    return False
def get_python_proficiency(skills):
    """
    Find Python proficiency level from skills list.
    Return proficiency level or None if not found.
    """
    for skill in skills:
        if skill.get('name', '').lower() == 'python':
            return skill.get('proficiency', None)
    return None
def calculate_ai_ml_years(career_history):
    """
    Count years spent in AI/ML roles.
    Search job descriptions for AI/ML keywords.
    """
    ai_ml_keywords = ['ml', 'ai', 'machine learning', 'deep learning', 'nlp', 'llm', 'transformer', 'neural', 'data science']
    
    total_ai_ml_months = 0
    
    for job in career_history:
        description = (job.get('description') or '').lower()
        
        # Check if this job is AI/ML related
        is_ai_ml_job = any(keyword in description for keyword in ai_ml_keywords)
        
        if is_ai_ml_job:
            # Add the duration of this job to total
            duration = job.get('duration_months', 0)
            total_ai_ml_months += duration
    
    # Convert months to years
    return round(total_ai_ml_months / 12, 1)
# Path to your JSONL file
jsonl_file = "candidates.jsonl"

# This will store all candidates as rows
all_candidates_rows = []

# Open the JSONL file
with open(jsonl_file, 'r') as f:
    
    for line_num, line in enumerate(f):
        # Read each line and parse it as JSON
        candidate = json.loads(line)
        
        # Extract the fields you need (you decide which ones!)
        row = {
            'candidate_id': candidate['candidate_id'],
            'name': candidate['profile']['anonymized_name'],
            'headline': candidate['profile']['headline'],
            'years_of_experience': candidate['profile']['years_of_experience'],
            'current_title': candidate['profile']['current_title'],
            'current_company': candidate['profile']['current_company'],
            'location': candidate['profile']['location'],
            'country': candidate['profile']['country'],
            'recruiter_response_rate': candidate['redrob_signals'].get('recruiter_response_rate'),
            'notice_period_days': candidate['redrob_signals'].get('notice_period_days'),
            'last_active_date': candidate['redrob_signals'].get('last_active_date'),
            'github_activity_score': candidate['redrob_signals'].get('github_activity_score'),
            'willing_to_relocate': candidate['redrob_signals'].get('willing_to_relocate'),
            # NEW TECHNICAL SIGNALS:
            'has_ranking_retrieval_exp': search_career_history_for_keywords(
            candidate['career_history'], 
            ['ranking', 'retrieval', 'search', 'matching', 'recommendation', 'rag']
            ),
            'has_embeddings_exp': (
            search_career_history_for_keywords(
            candidate['career_history'],
            ['embedding', 'vector', 'semantic']
            ) or
            search_skills_for_keywords(candidate['skills'], ['embedding', 'vector'])
            ),
            'has_vector_db_exp': search_skills_for_keywords(
            candidate['skills'],
            ['pinecone', 'weaviate', 'qdrant', 'milvus', 'faiss']
            ), 
            'num_jobs': len(candidate['career_history']),
            'num_skills': len(candidate['skills']),
            'ai_ml_years': calculate_ai_ml_years(candidate['career_history']),
            'python_proficiency': get_python_proficiency(candidate['skills']),
        }    
            

            
        

        
        # Add this row to your list
        all_candidates_rows.append(row)
        
        # Progress indicator (print every 20,000 candidates)
        if line_num % 20000 == 0:
            print(f"Processed {line_num} candidates...")

# Convert list of rows to a DataFrame (table)
df = pd.DataFrame(all_candidates_rows)

# Save to CSV
df.to_csv('candidates_simple.csv', index=False)
print("✅ Saved to candidates_simple.csv")

# Show first 5 rows
print("\nFirst 5 rows:")
print(df.head())

Processed 0 candidates...
Processed 20000 candidates...
Processed 40000 candidates...
Processed 60000 candidates...
Processed 80000 candidates...
✅ Saved to candidates_simple.csv

First 5 rows:
   candidate_id          name                                       headline  \
0  CAND_0000001      Ira Vora           Backend Engineer | SQL, Spark, Cloud   
1  CAND_0000002  Saanvi Sethi      Operations Manager | 12.5+ yrs experience   
2  CAND_0000003  Yash Agarwal         Customer Support | 1.1+ yrs experience   
3  CAND_0000004     Anil Bose  Marketing Manager | Driving business outcomes   
4  CAND_0000005   Aisha Sethi               Accountant | Helping teams scale   

   years_of_experience       current_title   current_company  \
0                  6.9    Backend Engineer          Mindtree   
1                 12.5  Operations Manager             Wipro   
2                  1.1    Customer Support               TCS   
3                  3.8   Marketing Manager    Dunder Mifflin   
4    

In [59]:
df=pd.read_csv("candidates_simple.csv")

In [60]:
df

,candidate_id,name,headline,years_of_experience,current_title,current_company,location,country,recruiter_response_rate,notice_period_days,last_active_date,github_activity_score,willing_to_relocate,has_ranking_retrieval_exp,has_embeddings_exp,has_vector_db_exp,num_jobs,num_skills,ai_ml_years,python_proficiency
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",6.9,Backend Engineer,Mindtree,Toronto,Canada,0.34,60,2026-05-20,9.2,False,False,False,True,2,17,6.8,NaN
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,12.5,Operations Manager,Wipro,"Chennai, Tamil Nadu",India,0.29,60,2025-11-12,-1.0,False,True,False,False,4,9,8.1,NaN
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,1.1,Customer Support,TCS,Austin,USA,0.46,150,2026-03-21,-1.0,False,False,False,False,1,6,1.1,NaN
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,3.8,Marketing Manager,Dunder Mifflin,Sydney,Australia,0.26,120,2026-03-25,-1.0,True,True,False,False,3,10,2.5,NaN
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,11.0,Accountant,Stark Industries,"Gurgaon, Haryana",India,0.37,30,2025-10-01,-1.0,True,False,False,False,4,6,7.1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,CAND_0099996,Myra Sethi,Operations Manager | 13.4+ yrs experience,13.4,Operations Manager,Wipro,"Ahmedabad, Gujarat",India,0.45,30,2025-12-04,-1.0,False,True,False,False,5,10,7.0,NaN
99996,CAND_0099997,Priya Krishnan,Mechanical Engineer | 10.6+ yrs experience,10.6,Mechanical Engineer,Hooli,"Bangalore, Karnataka",India,0.41,30,2026-03-08,44.1,False,True,False,False,4,7,7.6,NaN
99997,CAND_0099998,Shreya Agarwal,Analytics Engineer | 7.3+ yrs in data engineering,7.3,Analytics Engineer,InMobi,"Kochi, Kerala",India,0.33,90,2026-03-07,10.8,True,False,False,True,3,10,7.2,NaN
99998,CAND_0099999,Diya Menon,.NET Developer | Backend systems & APIs,4.3,.NET Developer,HCL,"Bhubaneswar, Odisha",India,0.46,120,2025-12-23,-1.0,False,False,False,True,2,7,0.0,NaN


In [61]:
df.count()

candidate_id                 100000
name                         100000
headline                     100000
years_of_experience          100000
current_title                100000
current_company              100000
location                     100000
country                      100000
recruiter_response_rate      100000
notice_period_days           100000
last_active_date             100000
github_activity_score        100000
willing_to_relocate          100000
has_ranking_retrieval_exp    100000
has_embeddings_exp           100000
has_vector_db_exp            100000
num_jobs                     100000
num_skills                   100000
ai_ml_years                  100000
python_proficiency             1378
dtype: int64

In [62]:
df.head(2)

,candidate_id,name,headline,years_of_experience,current_title,current_company,location,country,recruiter_response_rate,notice_period_days,last_active_date,github_activity_score,willing_to_relocate,has_ranking_retrieval_exp,has_embeddings_exp,has_vector_db_exp,num_jobs,num_skills,ai_ml_years,python_proficiency
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",6.9,Backend Engineer,Mindtree,Toronto,Canada,0.34,60,2026-05-20,9.2,False,False,False,True,2,17,6.8,NaN
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,12.5,Operations Manager,Wipro,"Chennai, Tamil Nadu",India,0.29,60,2025-11-12,-1.0,False,True,False,False,4,9,8.1,NaN


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   candidate_id               100000 non-null  object 
 1   name                       100000 non-null  object 
 2   headline                   100000 non-null  object 
 3   years_of_experience        100000 non-null  float64
 4   current_title              100000 non-null  object 
 5   current_company            100000 non-null  object 
 6   location                   100000 non-null  object 
 7   country                    100000 non-null  object 
 8   recruiter_response_rate    100000 non-null  float64
 9   notice_period_days         100000 non-null  int64  
 10  last_active_date           100000 non-null  object 
 11  github_activity_score      100000 non-null  float64
 12  willing_to_relocate        100000 non-null  bool   
 13  has_ranking_retrieval_exp  100

In [64]:
df.isnull().sum()

candidate_id                     0
name                             0
headline                         0
years_of_experience              0
current_title                    0
current_company                  0
location                         0
country                          0
recruiter_response_rate          0
notice_period_days               0
last_active_date                 0
github_activity_score            0
willing_to_relocate              0
has_ranking_retrieval_exp        0
has_embeddings_exp               0
has_vector_db_exp                0
num_jobs                         0
num_skills                       0
ai_ml_years                      0
python_proficiency           98622
dtype: int64

In [65]:
print(df.duplicated(df).sum())

0


In [66]:
# Filter for 5-9 years experience
target_exp = df[(df['years_of_experience'] >= 5) & (df['years_of_experience'] <= 9)]

print(f"Total candidates: {len(df)}")
print(f"Candidates with 5-9 years experience: {len(target_exp)}")
print(f"Percentage: {len(target_exp)/len(df)*100:.1f}%")

Total candidates: 100000
Candidates with 5-9 years experience: 34375
Percentage: 34.4%


In [67]:
target_exp = df[(df['years_of_experience'] >= 5) & (df['years_of_experience'] <= 9)]
has_both = target_exp[target_exp['has_ranking_retrieval_exp'] == True]

print(f"With 5-9 years: {len(target_exp)}")
print(f"With ranking/retrieval exp: {len(has_both)}")
print(f"Percentage: {len(has_both)/len(target_exp)*100:.1f}%")

With 5-9 years: 34375
With ranking/retrieval exp: 10672
Percentage: 31.0%


In [68]:
# Start with our filtered pool
target_exp = df[(df['years_of_experience'] >= 5) & (df['years_of_experience'] <= 9)]
has_ranking = target_exp[target_exp['has_ranking_retrieval_exp'] == True]

# Now add vector DB or embeddings requirement
has_tech_stack = has_ranking[
    (has_ranking['has_vector_db_exp'] == True) | (has_ranking['has_embeddings_exp'] == True)
]

print(f"With ranking/retrieval exp: {len(has_ranking)}")
print(f"  → Among them, with vector_db OR embeddings: {len(has_tech_stack)}")
print(f"  → Percentage: {len(has_tech_stack)/len(has_ranking)*100:.1f}%")

With ranking/retrieval exp: 10672
  → Among them, with vector_db OR embeddings: 1721
  → Percentage: 16.1%


In [ ]:
# Our filtered pool so far
target_exp = df[(df['years_of_experience'] >= 5) & (df['years_of_experience'] <= 9)]
has_ranking = target_exp[target_exp['has_ranking_retrieval_exp'] == True]
has_tech = has_ranking[(has_ranking['has_vector_db_exp'] == True) | (has_ranking['has_embeddings_exp'] == True)]

# Now check recruiter response rate > 0.5 (50%)
responsive = has_tech[has_tech['recruiter_response_rate'] > 0.5]

print(f"With ranking + tech stack: {len(has_tech)}")
print(f"  → Among them, with recruiter_response_rate > 0.5: {len(responsive)}")
print(f"  → Percentage: {len(responsive)/len(has_tech)*100:.1f}%")
print(f"\nAverage recruiter response rate in this pool: {has_tech['recruiter_response_rate'].mean():.2f}")

With ranking + tech stack: 1721
  → Among them, with recruiter_response_rate > 0.5: 850
  → Percentage: 49.4%

Average recruiter response rate in this pool: 0.49


In [72]:
from datetime import datetime, timedelta
# Our filtered pool
target_exp = df[(df['years_of_experience'] >= 5) & (df['years_of_experience'] <= 9)]
has_ranking = target_exp[target_exp['has_ranking_retrieval_exp'] == True]
has_tech = has_ranking[(has_ranking['has_vector_db_exp'] == True) | (has_ranking['has_embeddings_exp'] == True)]
responsive = has_tech[has_tech['recruiter_response_rate'] > 0.5]

# Convert last_active_date to datetime
responsive['last_active_date'] = pd.to_datetime(responsive['last_active_date'])

# Check who was active in last 3 months (recent)
cutoff_date = datetime(2026, 3, 19)  # 3 months before June 19, 2026
recently_active = responsive[responsive['last_active_date'] > cutoff_date]

print(f"Responsive candidates: {len(responsive)}")
print(f"  → Among them, recently active (< 3 months): {len(recently_active)}")
print(f"  → Percentage: {len(recently_active)/len(responsive)*100:.1f}%")

Responsive candidates: 850
  → Among them, recently active (< 3 months): 390
  → Percentage: 45.9%


C:\Users\Abhishek Jadhao\AppData\Local\Temp\ipykernel_24224\957459976.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  responsive['last_active_date'] = pd.to_datetime(responsive['last_active_date'])


In [79]:
# Function to calculate scores for each component
def calculate_exp_score(years):
    """Score based on years of experience (5-9 is best = 100)"""
    if 5 <= years <= 9:
        return 100
    elif 4 <= years < 5:
        return 80
    elif 9 < years <= 10:
        return 80
    elif 3 <= years < 4:
        return 60
    elif 10 < years <= 12:
        return 60
    else:
        return 0

def calculate_ai_ml_score(ai_ml_years):
    """Score based on AI/ML years (5+ is best = 100)"""
    if ai_ml_years >= 5:
        return 100
    elif ai_ml_years >= 4:
        return 80
    elif ai_ml_years >= 3:
        return 60
    elif ai_ml_years >= 2:
        return 40
    else:
        return 0

def calculate_ranking_score(has_ranking):
    """Score for ranking/retrieval experience (binary)"""
    return 100 if has_ranking else 0

def calculate_tech_score(has_vector_db, has_embeddings):
    """Score for vector DB or embeddings (binary)"""
    if has_vector_db or has_embeddings:
        return 100
    else:
        return 0

def calculate_behavioral_score(response_rate):
    """Score based on recruiter response rate (0-1 to 0-100)"""
    if pd.isna(response_rate):
        return 0
    return response_rate * 100

def calculate_location_score(country, willing_to_relocate):
    """Score based on location"""
    if country == 'India':
        return 100
    elif willing_to_relocate:
        return 80
    else:
        return 20

# Calculate all scores
df['exp_score'] = df['years_of_experience'].apply(calculate_exp_score)
df['ai_ml_score'] = df['ai_ml_years'].apply(calculate_ai_ml_score)
df['ranking_score'] = df['has_ranking_retrieval_exp'].apply(calculate_ranking_score)
df['tech_score'] = df.apply(lambda row: calculate_tech_score(row['has_vector_db_exp'], row['has_embeddings_exp']), axis=1)
df['behavioral_score'] = df['recruiter_response_rate'].apply(calculate_behavioral_score)
df['location_score'] = df.apply(lambda row: calculate_location_score(row['country'], row['willing_to_relocate']), axis=1)

# Calculate final score
df['final_score'] = (
    (df['exp_score'] * 0.15) +
    (df['ai_ml_score'] * 0.15) +
    (df['ranking_score'] * 0.25) +
    (df['tech_score'] * 0.20) +
    (df['behavioral_score'] * 0.15) +
    (df['location_score'] * 0.10)
)

# ===== NEW: FILTER FOR AI/ML TITLES =====
ai_ml_keywords = ['ai', 'ml', 'machine learning', 'data scientist', 'nlp', 'computer vision', 
                   'deep learning', 'search engineer', 'ranking', 'recommendation', 'ai engineer',
                   'ml engineer', 'ai specialist', 'analytics engineer', 'research engineer']

# Create lowercase title column for matching
df['current_title_lower'] = df['current_title'].str.lower()

# Check if title contains AI/ML keywords
df['is_ai_ml_title'] = df['current_title_lower'].apply(
    lambda title: any(keyword in title for keyword in ai_ml_keywords) if pd.notna(title) else False
)

# Filter to keep only AI/ML related titles
df_filtered = df[df['is_ai_ml_title'] == True].copy()

print(f"Total candidates: {len(df)}")
print(f"Candidates with AI/ML titles: {len(df_filtered)}")
print(f"Filtered out: {len(df) - len(df_filtered)}")

# Get top 100
top_100 = df_filtered.nlargest(100, 'final_score')

# Add rank column
top_100 = top_100.reset_index(drop=True)
top_100['rank'] = range(1, len(top_100) + 1)

# Display results
print("\n" + "=" * 100)
print("TOP 100 CANDIDATES (AI/ML Titles Only)")
print("=" * 100)
print(top_100[['rank', 'candidate_id', 'name', 'current_title', 'current_company', 'years_of_experience', 'final_score']].head(20).to_string())

# Save to CSV
top_100.to_csv('top_100_ranked_filtered.csv', index=False)
print(f"\n✅ Saved to top_100_ranked_filtered.csv")
print(f"✅ Total ranked: {len(top_100)} candidates")

Total candidates: 100000
Candidates with AI/ML titles: 1939
Filtered out: 98061

TOP 100 CANDIDATES (AI/ML Titles Only)
    rank  candidate_id             name                    current_title current_company  years_of_experience  final_score
0      1  CAND_0010603      Aisha Desai                      ML Engineer          BYJU'S                  5.3        99.10
1      2  CAND_0030031       Anil Joshi                      AI Engineer       Microsoft                  5.7        99.10
2      3  CAND_0061265     Tanya Chopra  Recommendation Systems Engineer            Zoho                  6.6        99.10
3      4  CAND_0064326     Nisha Pillai                  Search Engineer       Sarvam AI                  7.6        99.10
4      5  CAND_0070525        Ved Kumar    Senior Software Engineer (ML)  Mad Street Den                  5.4        98.95
5      6  CAND_0006418  Rahul Mukherjee        Machine Learning Engineer      Verloop.io                  5.7        98.80
6      7  CAND_0073

In [80]:
top_100.head(10)

,candidate_id,name,headline,years_of_experience,current_title,current_company,location,country,recruiter_response_rate,notice_period_days,...,exp_score,ai_ml_score,ranking_score,tech_score,behavioral_score,location_score,final_score,current_title_lower,is_ai_ml_title,rank
0,CAND_0010603,Aisha Desai,ML Engineer | Building ML-powered solutions,5.3,ML Engineer,BYJU'S,"Bangalore, Karnataka",India,0.94,90,...,100,100,100,100,94.0,100,99.10,ml engineer,True,1
1,CAND_0030031,Anil Joshi,"AI Engineer | ML, NLP, Recommendation Systems",5.7,AI Engineer,Microsoft,"Trivandrum, Kerala",India,0.94,30,...,100,100,100,100,94.0,100,99.10,ai engineer,True,2
2,CAND_0061265,Tanya Chopra,"Recommendation Systems Engineer | Search, Rank...",6.6,Recommendation Systems Engineer,Zoho,"Delhi, Delhi",India,0.94,120,...,100,100,100,100,94.0,100,99.10,recommendation systems engineer,True,3
3,CAND_0064326,Nisha Pillai,"Search Engineer | Search, Ranking & Retrieval",7.6,Search Engineer,Sarvam AI,"Gurgaon, Haryana",India,0.94,45,...,100,100,100,100,94.0,100,99.10,search engineer,True,4
4,CAND_0070525,Ved Kumar,Senior Software Engineer (ML) | Building ML-po...,5.4,Senior Software Engineer (ML),Mad Street Den,"Hyderabad, Telangana",India,0.93,45,...,100,100,100,100,93.0,100,98.95,senior software engineer (ml),True,5
5,CAND_0006418,Rahul Mukherjee,"Machine Learning Engineer | Search, Ranking & ...",5.7,Machine Learning Engineer,Verloop.io,"Gurgaon, Haryana",India,0.92,60,...,100,100,100,100,92.0,100,98.80,machine learning engineer,True,6
6,CAND_0073883,Zara Sen,AI Specialist | Data Science & ML enthusiast,5.3,AI Specialist,Krutrim,"Chandigarh, Chandigarh",India,0.92,90,...,100,100,100,100,92.0,100,98.80,ai specialist,True,7
7,CAND_0000031,Ela Singh,"Recommendation Systems Engineer | Search, Rank...",6.0,Recommendation Systems Engineer,Swiggy,"Hyderabad, Telangana",India,0.91,60,...,100,100,100,100,91.0,100,98.65,recommendation systems engineer,True,8
8,CAND_0079064,Sai Saxena,"Senior Data Scientist | ML, NLP, Recommendatio...",5.2,Senior Data Scientist,Niramai,"Noida, Uttar Pradesh",India,0.91,120,...,100,100,100,100,91.0,100,98.65,senior data scientist,True,9
9,CAND_0036184,Riya Chopra,Recommendation Systems Engineer | Applied ML |...,6.0,Recommendation Systems Engineer,CRED,"Trivandrum, Kerala",India,0.90,30,...,100,100,100,100,90.0,100,98.50,recommendation systems engineer,True,10


In [81]:
top_100.tail(5)

,candidate_id,name,headline,years_of_experience,current_title,current_company,location,country,recruiter_response_rate,notice_period_days,...,exp_score,ai_ml_score,ranking_score,tech_score,behavioral_score,location_score,final_score,current_title_lower,is_ai_ml_title,rank
95,CAND_0014683,Ritu Tiwari,Senior Software Engineer (ML) | Data Science &...,5.6,Senior Software Engineer (ML),Rephrase.ai,"Jaipur, Rajasthan",India,0.70,120,...,100,100,100,100,70.0,100,95.50,senior software engineer (ml),True,96
96,CAND_0078492,Aadhya Vora,Recommendation Systems Engineer | Applied ML |...,5.1,Recommendation Systems Engineer,Verloop.io,"Kochi, Kerala",India,0.70,30,...,100,100,100,100,70.0,100,95.50,recommendation systems engineer,True,97
97,CAND_0083307,Neha Patel,"Search Engineer | Search, Ranking & Retrieval",7.8,Search Engineer,CRED,"Vizag, Andhra Pradesh",India,0.70,120,...,100,100,100,100,70.0,100,95.50,search engineer,True,98
98,CAND_0014063,Ela Chowdary,AI Specialist | Building ML-powered solutions,5.4,AI Specialist,Sarvam AI,"Vizag, Andhra Pradesh",India,0.69,45,...,100,100,100,100,69.0,100,95.35,ai specialist,True,99
99,CAND_0029135,Yash Bose,AI Specialist | 5.8 yrs in analytics & ML,5.8,AI Specialist,Meesho,"Ahmedabad, Gujarat",India,0.69,60,...,100,100,100,100,69.0,100,95.35,ai specialist,True,100


In [ ]:
print(f"Total candidates: {len(df)}")
print(f"Candidates with AI/ML titles: {len(df_filtered)}")
print(f"Filtered out: {len(df) - len(df_filtered)}")

Total candidates: 100000
Candidates with AI/ML titles: 1939
Filtered out: 98061


In [ ]:
# Load top 100 ranked candidates
top_100 = pd.read_csv('top_100_ranked_filtered.csv')

def generate_reasoning(row):
    """
    Generate 1-2 line reasoning based on candidate's signals
    """
    reasons = []
    
    # Check experience
    years = row['years_of_experience']
    ai_ml_years = row['ai_ml_years']
    if 5 <= years <= 9 and ai_ml_years >= 5:
        reasons.append(f"{years:.1f} years experience with {ai_ml_years:.1f} in AI/ML")
    
    # Check technical skills
    tech_signals = []
    if row['has_ranking_retrieval_exp']:
        tech_signals.append("ranking/retrieval systems")
    if row['has_vector_db_exp']:
        tech_signals.append("vector databases")
    if row['has_embeddings_exp']:
        tech_signals.append("embeddings & semantic search")
    
    if tech_signals:
        reasons.append(f"Built {' + '.join(tech_signals)}")
    
    # Check behavioral signals
    if row['recruiter_response_rate'] > 0.7:
        reasons.append(f"{int(row['recruiter_response_rate']*100)}% recruiter engagement")
    
    # Check location
    if row['country'] == 'India':
        reasons.append("Based in India")
    elif row['willing_to_relocate']:
        reasons.append("Willing to relocate")
    
    # Check notice period
    if row['notice_period_days'] <= 60:
        reasons.append(f"Can join in {int(row['notice_period_days'])} days")
    
    # Combine reasons into 1-2 lines
    reasoning = " | ".join(reasons[:3])  # Take top 3 signals
    return reasoning

# Generate reasoning for each candidate
top_100['reasoning'] = top_100.apply(generate_reasoning, axis=1)

# Create final submission CSV with required columns
submission = top_100[['candidate_id', 'rank', 'final_score', 'reasoning']].copy()
submission = submission.rename(columns={'final_score': 'score'})

# Reorder columns
submission = submission[['candidate_id', 'rank', 'score', 'reasoning']]

# Display first 10
print("=" * 120)
print("FINAL SUBMISSION (Top 10)")
print("=" * 120)
print(submission.head(10).to_string(index=False))

# Save to CSV
submission.to_csv('final_submission.csv', index=False)
print(f"\n✅ Saved to final_submission.csv")
print(f"✅ Total candidates ranked: {len(submission)}")

# Show sample reasoning
print("\n" + "=" * 120)
print("SAMPLE REASONING:")
print("=" * 120)
for idx, row in submission.head(5).iterrows():
    print(f"Rank {row['rank']}: {row['reasoning']}")

FINAL SUBMISSION (Top 10)
candidate_id  rank  score                                                                                                                                             reasoning
CAND_0010603     1  99.10                    5.3 years experience with 5.2 in AI/ML | Built ranking/retrieval systems + embeddings & semantic search | 94% recruiter engagement
CAND_0030031     2  99.10 5.7 years experience with 5.6 in AI/ML | Built ranking/retrieval systems + vector databases + embeddings & semantic search | 94% recruiter engagement
CAND_0061265     3  99.10 6.6 years experience with 6.5 in AI/ML | Built ranking/retrieval systems + vector databases + embeddings & semantic search | 94% recruiter engagement
CAND_0064326     4  99.10 7.6 years experience with 7.6 in AI/ML | Built ranking/retrieval systems + vector databases + embeddings & semantic search | 94% recruiter engagement
CAND_0070525     5  98.95                    5.4 years experience with 5.3 in AI/ML | Built ra

In [ ]:
submission=pd.read_csv("final_submission.csv")

In [ ]:
submission.head(2)

,candidate_id,rank,score,reasoning
0,CAND_0010603,1,99.1,5.3 years experience with 5.2 in AI/ML | Built...
1,CAND_0030031,2,99.1,5.7 years experience with 5.6 in AI/ML | Built...


In [92]:
# Configure pandas to show full content
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Now read and print
submission = pd.read_csv('final_submission.csv')
print(submission.head(10))

   candidate_id  rank  score  \
0  CAND_0010603     1  99.10   
1  CAND_0030031     2  99.10   
2  CAND_0061265     3  99.10   
3  CAND_0064326     4  99.10   
4  CAND_0070525     5  98.95   
5  CAND_0006418     6  98.80   
6  CAND_0073883     7  98.80   
7  CAND_0000031     8  98.65   
8  CAND_0079064     9  98.65   
9  CAND_0036184    10  98.50   

                                                                                                                                               reasoning  
0                     5.3 years experience with 5.2 in AI/ML | Built ranking/retrieval systems + embeddings & semantic search | 94% recruiter engagement  
1  5.7 years experience with 5.6 in AI/ML | Built ranking/retrieval systems + vector databases + embeddings & semantic search | 94% recruiter engagement  
2  6.6 years experience with 6.5 in AI/ML | Built ranking/retrieval systems + vector databases + embeddings & semantic search | 94% recruiter engagement  
3  7.6 years experience wit

In [93]:
len(submission)

100